In [1]:
import importlib
import Analyser_ as hsv
importlib.reload(hsv)

from IPython.display import HTML, display
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Dark theme
display(HTML("""
<style>
body, .jp-Notebook, .jp-OutputArea-output, .jp-RenderedHTMLCommon {
    background-color: #1e1e1e !important;
    color: #d4d4d4 !important;
}
h2, h3, h4 { color: #4fc3f7 !important; }
.highlight {
    background-color: #2d2d2d !important;
    padding: 10px;
    border-left: 4px solid #3498db;
    margin: 4px 0;
    color: #d4d4d4;
}
</style>
"""))

print("Analyser imported. Dark theme applied.")

Analyser imported. Dark theme applied.


In [2]:
EXP_ROOT = "/Volumes/Amirali/hidden_states/experiments/baseline_v5_001"
df_all = hsv.analyze_all_models(EXP_ROOT, False, True, True)

display(HTML("<h2>Comprehensive Analysis – All Models & Datasets</h2>"))
display(df_all)

[1/33] Processing EleutherAI/gpt-neo-125m / ISEAR...
  ✓ EleutherAI/gpt-neo-125m/ISEAR -> linkage issues
[2/33] Processing EleutherAI/gpt-neo-125m / goEmo...
  ✓ EleutherAI/gpt-neo-125m/goEmo -> linkage issues
[3/33] Processing FacebookAI/roberta-base / ISEAR...
  ✓ FacebookAI/roberta-base/ISEAR -> linkage issues
[4/33] Processing FacebookAI/roberta-base / goEmo...
  ✓ FacebookAI/roberta-base/goEmo -> linkage issues
[5/33] Processing HuggingFaceTB/SmolLM2-1.7B / goEmo...
  ✓ HuggingFaceTB/SmolLM2-1.7B/goEmo -> 6 missing files; linkage issues; 25 layer anomalies; 51200 dead neurons
[6/33] Processing HuggingFaceTB/SmolLM2-135M / ISEAR...
  ✓ HuggingFaceTB/SmolLM2-135M/ISEAR -> 1 missing files; integrity corrupted; linkage issues
[7/33] Processing HuggingFaceTB/SmolLM2-135M / goEmo...
  ✓ HuggingFaceTB/SmolLM2-135M/goEmo -> linkage issues
[8/33] Processing HuggingFaceTB/SmolLM2-360M / ISEAR...
  ✓ HuggingFaceTB/SmolLM2-360M/ISEAR -> integrity corrupted; linkage issues
[9/33] Processing Hu

In [ ]:
# Display table
display(df_all)

In [ ]:
# Generate plots
hsv.plot_summary(df_all)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df_all.empty:
    # 1. Completion percentage heatmap (models vs datasets)
    pivot = df_all.pivot_table(index='model', columns='dataset', values='completion_pct', aggfunc='mean')
    plt.figure(figsize=(14, 8))
    sns.heatmap(pivot, annot=True, fmt=".1f", cmap='viridis', cbar_kws={'label': 'Completion %'})
    plt.title('Completion Percentage by Model and Dataset', color='#4fc3f7')
    plt.tight_layout()
    plt.show()

    # 2. Probe readiness per model (split by dataset)
    plt.figure(figsize=(12, 6))
    sns.countplot(data=df_all, x='model', hue='probe_ready', palette='Set2')
    plt.xticks(rotation=45, ha='right')
    plt.title('Probe Readiness by Model', color='#4fc3f7')
    plt.legend(title='Probe Ready')
    plt.tight_layout()
    plt.show()

    # 3. Missing files per model
    missing = df_all.groupby('model')['missing_files'].sum().sort_values()
    plt.figure(figsize=(10, 6))
    sns.barplot(x=missing.values, y=missing.index, color='#f48fb1')
    plt.title('Total Missing Files per Model', color='#4fc3f7')
    plt.xlabel('Missing Files Count')
    plt.tight_layout()
    plt.show()

    # 4. Anomaly scatter (dead neurons vs outliers)
    plt.figure(figsize=(10, 6))
    scatter = plt.scatter(df_all['dead_neurons'], df_all['outliers'],
                          c=df_all['completion_pct'], cmap='coolwarm', s=80)
    plt.colorbar(scatter, label='Completion %')
    plt.xlabel('Dead Neurons')
    plt.ylabel('Outliers')
    plt.title('Anomaly Overview', color='#4fc3f7')
    plt.tight_layout()
    plt.show()

    # 5. Integrity status distribution
    plt.figure(figsize=(8, 5))
    sns.countplot(data=df_all, x='integrity_status', hue='integrity_status', palette='viridis', legend=False)
    plt.title('Integrity Hash Status', color='#4fc3f7')
    plt.tight_layout()
    plt.show()

    # 6. Family / architecture summary (if available)
    if 'family' in df_all.columns and df_all['family'].notna().any():
        plt.figure(figsize=(10, 6))
        sns.countplot(data=df_all, x='family', hue='probe_ready', palette='Set2')
        plt.xticks(rotation=45, ha='right')
        plt.title('Probe Readiness by Model Family', color='#4fc3f7')
        plt.tight_layout()
        plt.show()
else:
    print("No data found.")